# Preprocessing Data - Prediksi Risiko Stroke

Notebook ini berisi tahapan preprocessing data:
1. Load dataset
2. Handle missing values (N/A pada kolom bmi)
3. Encoding data kategorikal
4. Normalisasi data
5. Handle imbalanced dataset dengan SMOTE

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import os
import joblib

## 1. Load Dataset

In [2]:
df = pd.read_csv('stroke-data.csv')
print("Dataset shape:", df.shape)
print("\n5 data pertama:")
df.head()

Dataset shape: (5110, 12)

5 data pertama:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 2. Cek Missing Values

In [3]:
print("Cek missing values:")
print(df.isnull().sum())

print("\nCek N/A pada bmi:")
print("Jumlah N/A:", (df['bmi'] == 'N/A').sum())

Cek missing values:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

Cek N/A pada bmi:
Jumlah N/A: 0


## 3. Handle Missing Values
Mengisi nilai N/A pada kolom bmi dengan median

In [4]:
df['bmi'] = df['bmi'].replace('N/A', np.nan)
df['bmi'] = df['bmi'].astype(float)

median_bmi = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(median_bmi)

print(f"Median BMI: {median_bmi}")
print(f"Missing values setelah imputasi: {df['bmi'].isnull().sum()}")

Median BMI: 28.1
Missing values setelah imputasi: 0


## 4. Hapus Kolom ID
Kolom id tidak relevan untuk prediksi

In [5]:
df.drop(columns=['id'], inplace=True)
print("Kolom setelah dihapus id:")
print(df.columns.tolist())

Kolom setelah dihapus id:
['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']


## 5. Encoding Data Kategorikal

In [6]:
le = LabelEncoder()

kolom_kategori = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for kolom in kolom_kategori:
    df[kolom] = le.fit_transform(df[kolom])

print("Hasil Encoding:")
print("- gender        : Female=0, Male=1, Other=2")
print("- ever_married  : No=0, Yes=1")
print("- Residence_type: Rural=0, Urban=1")
print("- smoking_status: Unknown=0, formerly smoked=1, never smoked=2, smokes=3")
print("- work_type     : Govt_job=0, Never_worked=1, Private=2, Self-employed=3, children=4")

df.head()

Hasil Encoding:
- gender        : Female=0, Male=1, Other=2
- ever_married  : No=0, Yes=1
- Residence_type: Rural=0, Urban=1
- smoking_status: Unknown=0, formerly smoked=1, never smoked=2, smokes=3
- work_type     : Govt_job=0, Never_worked=1, Private=2, Self-employed=3, children=4


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1,67.0,0,1,1,2,1,228.69,36.6,1,1
1,0,61.0,0,0,1,3,0,202.21,28.1,2,1
2,1,80.0,0,1,1,2,0,105.92,32.5,2,1
3,0,49.0,0,0,1,2,1,171.23,34.4,3,1
4,0,79.0,1,0,1,3,0,174.12,24.0,2,1


## 6. Pisahkan Fitur dan Target

In [7]:
X = df.drop(columns=['stroke'])
y = df['stroke']

print(f"Fitur (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

print("\nDistribusi target:")
print(y.value_counts())

Fitur (X) shape: (5110, 10)
Target (y) shape: (5110,)

Distribusi target:
stroke
0    4861
1     249
Name: count, dtype: int64


## 7. Normalisasi Data
Menggunakan StandardScaler untuk fitur numerik

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data setelah normalisasi:")
print(f"Mean: {X_scaled.mean(axis=0).round(4)}")
print(f"Std:  {X_scaled.std(axis=0).round(4)}")

Data setelah normalisasi:
Mean: [-0.  0. -0.  0. -0. -0. -0.  0. -0.  0.]
Std:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 8. Handle Imbalanced Dataset dengan SMOTE

In [9]:
print("Distribusi sebelum SMOTE:")
print(pd.Series(y).value_counts())

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print("\nDistribusi setelah SMOTE:")
print(pd.Series(y_resampled).value_counts())

Distribusi sebelum SMOTE:
stroke
0    4861
1     249
Name: count, dtype: int64

Distribusi setelah SMOTE:
stroke
1    4861
0    4861
Name: count, dtype: int64


## 9. Simpan Data Preprocessed

In [10]:
os.makedirs('data', exist_ok=True)
os.makedirs('model', exist_ok=True)

np.save('data/X_train.npy', X_resampled)
np.save('data/y_train.npy', y_resampled)
joblib.dump(scaler, 'model/scaler.pkl')

print("Data preprocessing selesai!")
print(f"X_resampled shape: {X_resampled.shape}")
print(f"y_resampled shape: {y_resampled.shape}")

Data preprocessing selesai!
X_resampled shape: (9722, 10)
y_resampled shape: (9722,)
